In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [14]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 1


### Load Player Data and Bookmaker Data

In [16]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
df = s26

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Update projected starting lineups

In [17]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Applications/Documents/NBA-Prop-Predictor/MODELS/teamInfo.py
Updated 16 teams with confirmed lineups


### Top EVs for single bets

In [19]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 250) & (usData['ODDS'] >= -250)]

results = calculateSingleBets(df, singlePTSBookies, model, features, current_date, 
                             edge_threshold=0.20, stake=100, 
                             variance_inflation=1.1, distribution_type='t', 
                             use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)  

singleBets = results
singleBets = singleBets[(singleBets['SIGMA FLAG'] == 'Med') | (singleBets['SIGMA FLAG'] == 'Low')].sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']]
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Dyson Daniels,Bovada,12.5,9.43,Under,210,1,131.42,131.4,0.626,Med
1,Andrew Nembhard,Bovada,8.5,6.27,Under,210,1,110.71,110.7,0.527,Med
2,Dyson Daniels,Bovada,13.5,9.43,Under,160,1,109.30,109.3,0.683,Med
3,Jose Alvarado,Bovada,7.5,6.21,Under,215,1,92.75,92.7,0.431,Med
4,Dyson Daniels,Bovada,14.5,9.43,Under,125,1,91.41,91.4,0.731,Med


## Top EVs for 2 leg bets

### Underdog picks

In [20]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results
underdogPairs = underdogPairs[
    underdogPairs[['SIGMA FLAG 1', 'SIGMA FLAG 2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Jared McCain,Andrew Nembhard,7.5,11.5,under,under,1,8.58,0.429,Low,Med
1,Jose Alvarado,Andrew Nembhard,10.5,11.5,under,under,1,8.48,0.424,Med,Med
2,Jared McCain,Jose Alvarado,7.5,10.5,under,under,1,7.78,0.389,Low,Med
3,Keldon Johnson,Andrew Nembhard,10.5,11.5,under,under,1,7.21,0.360,Med,Med
4,Kris Murray,Andrew Nembhard,6.5,11.5,under,under,1,7.18,0.359,Low,Med


### Prizepicks picks

In [21]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)
pairsPrizepicks = results
pairsPrizepicks = pairsPrizepicks[
    pairsPrizepicks[['SIGMA FLAG 1', 'SIGMA FLAG 2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Dyson Daniels,Andrew Nembhard,15.5,11.5,under,under,1,10.29,0.514,Med,Med
1,Dyson Daniels,Jose Alvarado,15.5,10.5,under,under,1,9.41,0.471,Med,Med
2,Jose Alvarado,Andrew Nembhard,10.5,11.5,under,under,1,8.48,0.424,Med,Med
3,Dyson Daniels,Keldon Johnson,15.5,10.5,under,under,1,8.07,0.404,Med,Med
4,Dyson Daniels,Kris Murray,15.5,6.5,under,under,1,8.05,0.402,Med,Low


## 3 leg parlay

### Underdog picks

In [22]:


dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=10, 
                     variance_inflation=1.1, distribution_type='t', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg
underdogTrios = threeLeg[
    threeLeg[['SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Jared McCain,Jose Alvarado,Andrew Nembhard,7.5,10.5,11.5,under,under,under,1,17.07,0.341,Low,Med,Med
1,Jared McCain,Keldon Johnson,Andrew Nembhard,7.5,10.5,11.5,under,under,under,1,15.20,0.304,Low,Med,Med
2,Jared McCain,Kris Murray,Andrew Nembhard,7.5,6.5,11.5,under,under,under,1,15.17,0.303,Low,Low,Med
3,Keldon Johnson,Jose Alvarado,Andrew Nembhard,10.5,10.5,11.5,under,under,under,1,15.07,0.301,Med,Med,Med
4,Jose Alvarado,Kris Murray,Andrew Nembhard,10.5,6.5,11.5,under,under,under,1,15.03,0.301,Med,Low,Med


### Prizepicks picks

In [23]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='normal', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg[
    threeLeg[['SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Jared McCain,Jose Alvarado,Andrew Nembhard,7.5,10.5,11.5,under,under,under,1,17.07,0.341,Low,Med,Med
1,Jared McCain,Keldon Johnson,Andrew Nembhard,7.5,10.5,11.5,under,under,under,1,15.20,0.304,Low,Med,Med
2,Jared McCain,Kris Murray,Andrew Nembhard,7.5,6.5,11.5,under,under,under,1,15.17,0.303,Low,Low,Med
3,Keldon Johnson,Jose Alvarado,Andrew Nembhard,10.5,10.5,11.5,under,under,under,1,15.07,0.301,Med,Med,Med
4,Jose Alvarado,Kris Murray,Andrew Nembhard,10.5,6.5,11.5,under,under,under,1,15.03,0.301,Med,Low,Med


In [24]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'L-{window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Anthony Davis,Over,24.5,-137,2025-11-09,2025-11-08T20:19:27Z
2,PrizePicks,player_points,Cooper Flagg,Over,17.5,-137,2025-11-09,2025-11-08T20:19:27Z
4,PrizePicks,player_points,Alex Sarr,Over,17.5,-137,2025-11-09,2025-11-08T20:19:27Z
6,PrizePicks,player_points,P.J. Washington,Over,16.5,-137,2025-11-09,2025-11-08T20:19:27Z
8,PrizePicks,player_points,C.J. McCollum,Over,15.5,-137,2025-11-09,2025-11-08T20:19:27Z
...,...,...,...,...,...,...,...,...
2208,PrizePicks,player_blocks_steals,Grayson Allen,Over,1.5,-137,2025-11-09,2025-11-08T20:20:38Z
2210,PrizePicks,player_blocks_steals,Ivica Zubac,Over,1.5,-137,2025-11-09,2025-11-08T20:20:38Z
2212,PrizePicks,player_blocks_steals,Collin Gillespie,Over,1.5,-137,2025-11-09,2025-11-08T20:20:38Z
2214,PrizePicks,player_blocks_steals,Kris Dunn,Over,1.5,-137,2025-11-09,2025-11-08T20:20:38Z


In [25]:
line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 99 records for player_points to player_points.csv
Saved 54 records for player_rebounds to player_rebounds.csv
Saved 29 records for player_assists to player_assists.csv
Saved 14 records for player_threes to player_threes.csv
Saved 6 records for player_blocks to player_blocks.csv
Saved 11 records for player_steals to player_steals.csv
Saved 39 records for player_field_goals to player_field_goals.csv
Saved 17 records for player_frees_made to player_frees_made.csv
Saved 8 records for player_frees_attempts to player_frees_attempts.csv
Saved 100 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 101 records for player_points_rebounds to player_points_rebounds.csv
Saved 94 records for player_points_assists to player_points_assists.csv
Saved 46 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 12 records for player_turnovers to player_turnovers.csv
Saved 15 records for player_blocks_steals to player_blocks_steals.csv

All categor

In [26]:
underdog = dfsData[(dfsData['BOOKMAKER'] == 'Underdog')]
underdog= underdog.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
line_hit_data = []

for index, row in underdog.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

Saved 69 records for player_points to player_points.csv
Saved 29 records for player_rebounds to player_rebounds.csv
Saved 18 records for player_assists to player_assists.csv
Saved 11 records for player_threes to player_threes.csv
Saved 2 records for player_blocks to player_blocks.csv
Saved 4 records for player_steals to player_steals.csv
Saved 81 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 42 records for player_points_rebounds to player_points_rebounds.csv
Saved 30 records for player_points_assists to player_points_assists.csv
Saved 19 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 11 records for player_turnovers to player_turnovers.csv
Saved 2 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG
